<div dir="rtl">
<h1>اثر یک ورودی پس از چند بازنویسی حافظه</h1>
<p>درس 32 از 76 · پیش از Attention: یک شبکه چطور گذشته را نگه می‌داشت؟ · <code dir="ltr">26b-sequence-memory</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-05/chapter-01/26b-sequence-memory.html">📖 بازگشت به همین درس</a></p>
<p>حالت بازگشتی را بسازید و ماندگاری یک تغییر کوچک را اندازه بگیرید.</p><p>پیش‌نیاز: حلقهٔ Python و ضرب یک ضریب در حالت قبلی کافی است؛ این مدل خطی، LSTM نیست.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>در ورودی [1,0,0,0] با factor=0.5 کدام گام بیشترین اثر عدد ۱ را دارد؟ با انتقال ۱ به انتها چه می‌شود؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
sequence = [1.,0.,0.,0.]
print('input:', sequence, 'retention:', 0.5)

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>تابع recurrent_states(Sequence, factor, initial=0.0) را بنویسید. در هر گام h=factor*h+x و خروجی فهرست همهٔ حالت‌ها باشد؛ initial فقط پیش از حلقه استفاده شود.</p>
</div>

In [ ]:
def recurrent_states(sequence, factor, initial=0.0):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = recurrent_states(sequence,0.5)
    if result is None: return False
    assert result == [1.,0.5,0.25,0.125]
    assert recurrent_states([0.,0.,0.,1.],0.5)[-1] == 1.
    assert recurrent_states([2.,-1.],2.,initial=3.) == [8.,15.]
    assert recurrent_states([],0.5) == []
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط factor را میان 0.5، 1 و 1.5 تغییر دهید؛ طول ۲۰ و تغییر حالت آغازین ثابت‌اند. این آزمایش دربارهٔ انتقال اثر است، نه کیفیت فهم متن.</p>
</div>

In [ ]:
for factor in (0.5,1.,1.5):
    a,b = 0.,1.
    for _ in range(20):
        a,b = factor*a, factor*b
    print(factor, 'initial-state difference after 20 steps:', b-a)

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>در کد خراب، h هر بار داخل حلقه صفر می‌شود. تابع remember(Sequence, factor) را اصلاح کنید؛ خروجی همهٔ حالت‌هاست و حالت آغازین صفر است.</p>
</div>

In [ ]:
wrong = []
for value in sequence:
    h = 0.
    h = 0.5*h + value
    wrong.append(h)
print('lost memory:', wrong)

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def remember(sequence, factor):
    # TODO
    return None

In [ ]:
def test_repair():
    result = remember(sequence,0.5)
    if result is None: return False
    assert result == [1.,0.5,0.25,0.125]
    assert remember([2.,1.,0.],1.) == [2.,3.,3.]
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>MiniGPT حالت RNN حمل نمی‌کند. در mini_gpt/attention.py هر Query از Valueهای موقعیت‌های مجاز ترکیب می‌سازد؛ این مقایسه علت انتخاب یک مسیر مستقیم‌تر به گذشته را روشن می‌کند.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>در این مثال اثر ورودی قدیمی از چه مسیری می‌گذرد؟ Attention چه مسیر متفاوتی می‌سازد و در عوض چه جدولی را نگه می‌دارد؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-05/chapter-01/26b-sequence-memory.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/26b-sequence-memory.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>